# Escore Modelling Notebook

This notebook is dedicated to fitting the Escore model. It requires building a training dataset made of *echotypes*. See the "Echotypes Extraction Notebook" for more information.

## Load data

In [1]:
import pandas as pd

df = pd.read_parquet("../private/echotypes.parquet")

## Visualizing echotypes data

In [2]:
from escore.utils.plots import plot_echotypes_dist_2D

plot_echotypes_dist_2D(
    df,
    x_channel="channel_1_Sv",
    y_channel="channel_2_Sv",
    ref_channel="channel_0_Sv",
    group_channel="echotype_id",
    xlabel="ΔMVBS 70-38kHz [dB]",
    ylabel="ΔMVBS 120-38kHz [dB]"
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'7f1acc0b-4a7b-489f-848e-84a99e2c9723': {'version…

## Classifying echotypes into *echoclasses*

The actual `EscoreModel` learns gaussian distributions in the $\Delta \text{MVBS}$ space for a number of classes. As such, the `EscoreModel` is a supervised model.

However, the echotypes selection steps provides 2 pieces of information:

1. The **$\Delta \text{MVBS}$ distribution is more clustered** as in the whole dataset. This is expected as the user should select echogram features with defined frequency-response, and avoid gradients of mixed $R(f)$.

2. Samples are **grouped by echotype**. This is information is non trivial as we expect the final classification to put all (or most) samples from a given echotype in the same class.

Echotypes are a grouping that is supposed to be contained within the desired echoclasses grouping. Two echotypes of the same echoclass only differ in their spatio-temporal position.

We then perform a clustering of echotypes in $\Delta \text{MVBS}$ space.

**Clustering approach**

We aggregate samples by echotypes (typically using mean) and cluster this much simple cloud of points. This clearly enforces the echotype grouping. However, some information on the sample-level distribution is lost.

### Define clustering pipeline

In [3]:
from escore.estimators import SvDifferenceExtractor
from sklearn.pipeline import Pipeline
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

N_CLASSES = 5

# Aggregate data by echotypes (not used here but will be applied each time)
data = df[["echotype_id", "channel_0_Sv", "channel_1_Sv", "channel_2_Sv"]].copy()
data_agg = data.groupby("echotype_id").mean()
X_agg = data_agg.copy()

# Build pipeline
preprocessor = Pipeline([
    ("features", SvDifferenceExtractor(ref_channel_idx=0)),
    ("scaler", StandardScaler()),
])
classifier = AgglomerativeClustering(linkage='ward', n_clusters=N_CLASSES)

clustering_pipeline = Pipeline([
    ("preproc", preprocessor),
    ("clf", classifier),
])

### Chose `n_clusters`

In [4]:
import hvplot.pandas
from escore.utils.evaluation import n_clusters_metrics, n_clusters_metrics_raw

# Scores on aggregated (mean) data
scores_agg = n_clusters_metrics(X_agg, preprocessor, classifier, n_clusters_range=range(2, 15))

# Scores on raw data (label of an echotype is given to all its samples)
scores_raw = n_clusters_metrics_raw(data, preprocessor, classifier, n_clusters_range=range(2, 15))

In [5]:
# Interactive plot
kwrgs = dict(x="n_clusters", y="value", groupby="metric", grid=True, width=600, legend='top_right')
scores_agg_plot = (scores_agg.hvplot.line(**kwrgs) * scores_agg.hvplot.scatter(label=f"Aggregated samples N={len(data_agg)}", **kwrgs))
scores_raw_plot = (scores_raw.hvplot.line(**kwrgs) * scores_raw.hvplot.scatter(label=f"Raw samples N={len(data)}", **kwrgs))

scores_agg_plot * scores_raw_plot

BokehModel(combine_events=True, render_bundle={'docs_json': {'7efb3fb9-239e-4249-975f-8bf3a6e0aaa1': {'version…

### Predict

In [6]:
# Set the chosen n_clusters
clustering_pipeline.set_params(clf__n_clusters=5)

# Aggregate
data = df[["echotype_id", "channel_0_Sv", "channel_1_Sv", "channel_2_Sv"]].copy()
data_agg = data.groupby("echotype_id").mean()
X_agg = data_agg.copy()

# Predict & associate to data
labels = clustering_pipeline.fit_predict(X_agg)
data_agg["echoclass_id"] = labels

# Associate echoclasses predictions to raw samples
df_wclasses = df.join(data_agg[["echoclass_id"]], on="echotype_id", how="left")

### Inspect results

We can first visualize the echoclasses with respect to the data that was used to create them (i.e. aggregated by echotypes):

In [7]:
data_agg["ΔMVBS 70-38kHz [dB]"] = data_agg["channel_1_Sv"] - data_agg["channel_0_Sv"]
data_agg["ΔMVBS 120-38kHz [dB]"] = data_agg["channel_2_Sv"] - data_agg["channel_0_Sv"]
data_agg["Echoclass"] = data_agg["echoclass_id"].astype("str")
data_agg = data_agg.sort_values(by="echoclass_id")

data_agg.hvplot.scatter(
    x="ΔMVBS 70-38kHz [dB]",
    y="ΔMVBS 120-38kHz [dB]",
    by="Echoclass",
    width=500,
    title="Mean echotype ΔMVBS",
    grid=True
)

:NdOverlay   [Echoclass]
   :Scatter   [ΔMVBS 70-38kHz [dB]]   (ΔMVBS 120-38kHz [dB])

Since the Escore model will learn the raw samples distribution of echoclasses, we can visualize it to after attributing to each sample the class of its parent echotype:

In [13]:
plot_echotypes_dist_2D(
    df_wclasses,
    x_channel="channel_1_Sv",
    y_channel="channel_2_Sv",
    ref_channel="channel_0_Sv",
    group_channel="echoclass_id",
    xlabel="ΔMVBS 70-38kHz [dB]",
    ylabel="ΔMVBS 120-38kHz [dB]"
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'5657985d-d72d-4778-80d3-10f5e591e230': {'version…

## Using *echoclasses* to fit an `EscoreModel` instance


In [14]:
X, y = df_wclasses[["channel_0_Sv", "channel_1_Sv", "channel_2_Sv"]], df_wclasses["echoclass_id"]